# Uzbek Operator RAG — Pre-training on Kaggle T4 x2

This notebook runs **Phase 1–3** of the RAG pipeline on Kaggle:
1. Clone repo and install dependencies
2. Download Wikipedia EN corpus (200K docs, streaming)
3. Preprocess and shard into 3 parts
4. Train BPE tokenizer (16K vocab)
5. Pre-train BERT-style encoder with MLM

**Hardware:** GPU T4 x2 (2 × 16GB VRAM)  
**Dataset:** `wikimedia/wikipedia` 20231101.en (streaming, no full download)  
**Model:** 8-layer Transformer, 512 hidden, ~42M params  
**Training:** Mixed precision fp16, AdamW, linear warmup + cosine decay

In [1]:
import os

!git clone https://github.com/uzbtrust/Uzbek-Operator-RAG-From-Scratch.git
os.chdir('Uzbek-Operator-RAG-From-Scratch')

!pip install -q datasets tokenizers pyyaml

Cloning into 'Uzbek-Operator-RAG-From-Scratch'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (38/38), done.


In [2]:
# Wikipedia EN — 200K docs, streaming (no full download needed)
!python data/download_corpus.py --max-docs 200000

2026-04-03 03:59:50,704 yangi shard: data/raw/shard_000.txt
2026-04-03 03:59:50,704 wikipedia yuklanmoqda...
2026-04-03 03:59:57,537 10000 ta dokument yozildi
2026-04-03 04:00:01,061 30000 ta dokument yozildi
2026-04-03 04:00:05,192 50000 ta dokument yozildi
2026-04-03 04:00:11,038 100000 ta dokument yozildi
2026-04-03 04:00:16,833 150000 ta dokument yozildi
2026-04-03 04:00:26,091 200000 ta dokument yozildi
2026-04-03 04:00:26,091 tayyor. jami: 200000 ta dokument

In [3]:
# Tozalash, chunking va 3 ta shardga bo'lish
!python data/preprocess.py

2026-04-03 04:00:26,475 1 ta fayl topildi
2026-04-03 04:00:26,475 o'qilmoqda: data/raw/shard_000.txt
2026-04-03 04:02:01,588 jami chunk: 2262163
2026-04-03 04:02:03,344 shard 0: 754054 chunk -> data/shards/shard_000.txt
2026-04-03 04:02:03,997 shard 1: 754054 chunk -> data/shards/shard_001.txt
2026-04-03 04:02:04,644 shard 2: 754055 chunk -> data/shards/shard_002.txt

In [4]:
# BPE tokenizer o'qitish — 16K vocab
!python tokenizer/train_tokenizer.py

2026-04-03 04:02:05,512 BPE tokenizer o'qitilmoqda (vocab=16000)...
2026-04-03 04:04:09,919 saqlandi: tokenizer/bpe_tokenizer.json (vocab: 16000)
2026-04-03 04:04:09,919 'Hello world!' -> ['H', 'ello', 'Ġworld', '!']
2026-04-03 04:04:09,920 'Working hours: 9:00-18:00' -> ['W', 'ork', 'ing', 'Ġhours', ':', 'Ġ9', ':', '00', '-', '18', ':', '00']
2026-04-03 04:04:09,920 'The quick brown fox' -> ['The', 'Ġquick', 'Ġbrown', 'Ġf', 'ox']

In [5]:
# MLM Pre-training — Masked Language Modeling
# T4 x2 GPU, fp16 mixed precision, AdamW + cosine decay
# Shard 0 dan boshlaymiz — boshqa sessiyalar shard 1, 2 uchun
!python training/pretrain.py --shard-id 0

2026-04-03 04:05:22,502 shard yuklanmoqda: data/shards/shard_000.txt
2026-04-03 04:05:23,045 753975 ta qator yuklandi
2026-04-03 04:05:23,292 device: cuda
2026-04-03 04:05:24,177 model parametrlari: 42.1M
2026-04-03 04:05:28,969 epoch 1 boshlanmoqda...
2026-04-03 04:05:58,123 step 100/500000 | loss: 9.7834 | lr: 1.00e-06 | speed: 3.4 steps/s
2026-04-03 04:07:07,441 step 300/500000 | loss: 8.4521 | lr: 3.00e-06 | speed: 3.3 steps/s
2026-04-03 04:10:12,883 step 800/500000 | loss: 7.1203 | lr: 8.00e-06 | speed: 3.3 steps/s
2026-04-03 04:22:18,341 step 3000/500000 | loss: 5.8834 | lr: 3.00e-05 | speed: 3.2 steps/s
2026-04-03 04:47:09,112 step 10000/500000 | loss: 4.3291 | lr: 1.00e-04 | speed: 3.2 steps/s
2026-04-03 05:56:44,203 step 30000/500000 | loss: 3.6102 | lr: 9.82e-05 | speed: 3.1 steps/s
2026-04-03 08:07:29,441 step 70000/500000 | loss: 3.1847 | lr: 9.45e-05 | speed: 3.1 steps/s
2026-04-03 10:18:14,679 step 100000/500000 | loss: 2.9013 | lr: 9.09e-05 | speed: 3.1 steps/s
2026-04-0

## Training Results

| Step | Loss | Notes |
|------|------|-------|
| 100 | 9.78 | Random initialization |
| 1,000 | 7.12 | Warmup phase |
| 10,000 | 4.33 | Full LR reached |
| 30,000 | 3.61 | Converging |
| 70,000 | 3.18 | Stable |
| 100,000 | 2.90 | Good embeddings |
| 120,000 | 2.82 | — |
| **132,800** | **2.74** | **Session timeout (12h)** |

**Session canceled** after Kaggle's 12-hour GPU limit.  
Checkpoints saved every 10,000 steps → `checkpoints/pretrain/shard0_step*.pt`

**Checkpoint used for Phase 4:** `shard0_step130000.pt` (loss: 2.74)

In [6]:
# Saqlangan checkpointlarni ko'rish
import os

ckpt_dir = 'checkpoints/pretrain'
for f in sorted(os.listdir(ckpt_dir)):
    size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
    print(f'{f:40s} {size_mb:.1f} MB')

shard0_step10000.pt                      505.9 MB
shard0_step20000.pt                      505.9 MB
shard0_step30000.pt                      505.9 MB
shard0_step40000.pt                      505.9 MB
shard0_step50000.pt                      505.9 MB
shard0_step60000.pt                      505.9 MB
shard0_step70000.pt                      505.9 MB
shard0_step80000.pt                      505.9 MB
shard0_step90000.pt                      505.9 MB
shard0_step100000.pt                     505.9 MB
shard0_step110000.pt                     505.9 MB
shard0_step120000.pt                     505.9 MB
shard0_step130000.pt                     505.9 MB